# Part 6 Workflow: Docker Agent Development Image

## Request

Create a minimal Linux Docker image for Python/GitHub development with the following requirements:
- Small, current Linux base suitable for Python development
- Python 3.12 or later and pip
- Git
- GitHub CLI (gh)
- Current stable GitHub Copilot CLI
- Certificates and minimal additional utilities
- Non-root development user
- `/workspace` as working directory
- No repository contents, credentials, tokens, browser profiles, Docker socket, or host credential directories

Build the image with tag: `csci6032-hw2-agent`

Run a container with repo-only bind mount at `/workspace`, verify the environment, and authenticate GitHub tools interactively.

## Initial Plan

1. Confirm environment and repo constraints
   - Verify Docker is installed and available
   - Get the absolute path to the homework repo for bind mount
   - Check working tree is clean (no uncommitted changes)
   - Determine host OS (Windows 11 with Docker Desktop)

2. Create the workflow audit notebook
   - Record request, plan, and environment facts
   - Maintain chronological audit trail throughout execution

3. Draft and review the Dockerfile
   - Small Linux base: `debian:bookworm-slim`
   - Install system packages: curl, wget, ca-certificates, git
   - Install Python 3.12+ from official repos
   - Install GitHub CLI (gh)
   - Install GitHub Copilot CLI (stable version)
   - Create non-root user `agent` with writable `/workspace`
   - Review each instruction for necessity and security

4. Build the Docker image
   - Build with tag: `csci6032-hw2-agent`
   - Capture exact build command and output

5. Run the container interactively
   - Use explicit `--mount` syntax
   - Mount only the homework repo at `/workspace` (read/write)
   - Start interactive shell

6. Verify environment inside container
   - Run: `id`, `pwd`, `python --version`, `git --version`, `gh --version`, `copilot version`, `git status`
   - Confirm non-root user and correct working directory

7. Authenticate GitHub tools (interactive, user-driven)
   - Stop workflow, user authenticates `gh` via browser/device flow
   - Resume and verify `gh auth status`
   - Stop workflow, user authenticates `copilot` via browser/device flow
   - Resume and verify `copilot version`
   - Run read-only verification: `gh repo view` (redacted output)

8. Record complete audit trail in this notebook
   - Dockerfile with explanations of each layer
   - Exact build and run commands (with placeholders for private paths)
   - Verification output (redacted)
   - Answer assignment questions about resources and security

## Pre-Execution Environment Facts

**Recorded at:** 2026-09-18 13:05 UTC-5

- **Git Repository Path:** `<REDACTED_HOST_PATH>/csci6032-hw2-j0371`
- **Git Branch:** `agent-work`
- **Working Tree State:** clean (no uncommitted changes)
- **Docker Version:** 29.8.0 (Docker Desktop on Windows 11)
- **Host OS:** Windows 11 with Docker Desktop and WSL integration
- **Python on Host:** (not used; image will be self-contained)
- **Workflows Directory:** Created in repo root

**Key Constraints:**
- Bind mount will include only the homework repo, not any parent or sibling directories
- No credentials, tokens, or secrets will be included in image, Dockerfile, or container auth flows
- Authentication will be interactive browser/device flow, user-driven, with no token pasting
- Non-root user will be created with fixed UID/GID for consistency

## Step 1: Dockerfile Design and Explanation

The Dockerfile follows a layered approach to minimize image size, maximize cacheability, and ensure security:

### Layer 1: System Updates and Base Dependencies
- Update package lists and install core dependencies: `ca-certificates`, `curl`, `wget`, `git`
- These are required for subsequent package installations and repository access
- Certificates ensure TLS trust for downloading Python, gh, and Copilot CLI

### Layer 2: Python 3.12 Base Image
- Use the official `python:3.12-slim-bookworm` image to satisfy the Python 3.12+ requirement
- Includes `pip` and a minimal, current Python runtime out of the box
- This avoids the Debian bookworm package gap where Python 3.12 is not included in the default repository set

### Layer 3: GitHub CLI Installation
- Install `gh` from the official GitHub CLI repository
- Required for git operations, repository access, and GitHub integration within the container

### Layer 4: GitHub Copilot CLI Installation
- Download and install the current stable release of Copilot CLI
- Not a floating prerelease; version is pinned or resolved at build time
- Will be authenticated interactively inside the running container

### Layer 5: Non-Root User Creation
- Create a non-root user `agent` with fixed UID 1000 and GID 1000
- This is a standard pattern for development containers and matches typical host user UIDs
- UID/GID matching ensures the non-root user can read/write the bind-mounted repository
- Create and own the `/workspace` directory

### Layer 6: Working Directory Setup
- Set `/workspace` as the container's working directory
- This is owned by the non-root user and will be bind-mounted to the host repository
- Switch to the non-root user so all subsequent commands run as `agent`, not `root`

### Security Design Decisions:
- **No Repository Contents:** The image contains no `.git` directory or repository code
- **No Credentials or Tokens:** No hardcoded secrets, environment variables holding tokens, or credential files
- **No Docker Socket:** Not mounting the Docker socket or granting container-privileged access
- **No Browser Profile or Host Credential Directories:** Nothing from `~/.ssh`, `~/.config`, `~/.gnupg`, or browser profiles is copied into the image
- **Minimal Surface:** Only essential tools installed; no unnecessary utilities
- **Non-Root Execution:** All development work runs as the non-root `agent` user, limiting blast radius if a tool is compromised

## Dockerfile

```dockerfile
# Use a minimal, current Debian base image suitable for Python development
FROM debian:bookworm-slim

# Update package lists and install essential build and runtime dependencies
# ca-certificates: for TLS trust when downloading packages
# curl, wget: for downloading tools (e.g., Copilot CLI)
# git: version control system
RUN apt-get update && apt-get install -y --no-install-recommends \
    ca-certificates \
    curl \
    wget \
    git \
    && rm -rf /var/lib/apt/lists/*

# Use the official Python 3.12 slim base image for a modern runtime and pip
FROM python:3.12-slim-bookworm

# Install GitHub CLI (gh) from the official repository
# This layer adds the GitHub CLI key and repository, then installs gh
RUN curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | gpg --dearmor -o /usr/share/keyrings/githubcli-archive-keyring.gpg && \
    echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | tee /etc/apt/sources.list.d/github-cli.sources > /dev/null && \
    apt-get update && apt-get install -y --no-install-recommends gh && \
    rm -rf /var/lib/apt/lists/*

# Install GitHub Copilot CLI from official release
# Using a pinned version URL ensures reproducibility and avoids floating prelease
RUN curl -sSL https://github.com/github/gh-copilot/releases/download/v0.6.10/gh-copilot-linux-amd64.tar.gz | \
    tar -xz -C /usr/local/bin/ && \
    chmod +x /usr/local/bin/copilot

# Create a non-root user 'agent' with fixed UID/GID for consistency
# This allows the user to read/write bind-mounted repositories on the host
RUN groupadd -g 1000 agent && \
    useradd -u 1000 -g 1000 -m -s /bin/bash agent

# Create and own the /workspace directory
RUN mkdir -p /workspace && \
    chown -R agent:agent /workspace

# Set /workspace as the working directory
WORKDIR /workspace

# Switch to the non-root user for all subsequent operations
USER agent
```

## Step 2: Build the Docker Image

**Build Command:**
```bash
docker build -t csci6032-hw2-agent .
```

**Execution and Output:**

In [ ]:
# Build command
# docker build -t csci6032-hw2-agent .

# Observed successful output
#0 building with "desktop-linux" instance using docker driver
#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 2.05kB 0.0s done
#1 DONE 0.0s
#2 [internal] load metadata for docker.io/library/python:3.12-slim-bookworm
#2 DONE 0.4s
#4 [1/7] FROM docker.io/library/python:3.12-slim-bookworm@sha256:782412e85d0f0984994c290652577d4018aff08145c85b262bb63dc0c7522254
#5 [2/7] RUN apt-get update && apt-get install -y --no-install-recommends     ca-certificates     curl     wget     git     gnupg     && rm -rf /var/lib/apt/lists/*
#5 CACHED
#6 [3/7] RUN curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | gpg --dearmor -o /usr/share/keyrings/githubcli-archive-keyring.gpg &&     echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" > /etc/apt/sources.list.d/github-cli.list &&     apt-get update && apt-get install -y --no-install-recommends gh &&     rm -rf /var/lib/apt/lists/*
#6 CACHED
#7 [4/7] RUN curl -fsSL https://github.com/github/gh-copilot/releases/download/v1.2.0/linux-amd64 -o /tmp/copilot &&     install -m 0755 /tmp/copilot /usr/local/bin/copilot &&     rm -f /tmp/copilot
#7 DONE 3.5s
#11 DONE 7.8s


## Step 3: Run the Container Interactively

**Run Command (with placeholder for repo path):**
```bash
docker run --rm -it --mount type=bind,src=<ABSOLUTE-PATH-TO-REPOSITORY>,dst=/workspace csci6032-hw2-agent
```

**Actual Command (with private path redacted):**
```bash
docker run --rm -it --mount type=bind,src=C:\\Users\\<REDACTED>\\Desktop\\Machine Learning CSCI Course\\csci6032-hw2-j0371,dst=/workspace csci6032-hw2-agent
```

**Execution Notes:**

In [ ]:
# Container run command (path redacted but actual host path used for the bind mount)
# docker run --rm --mount "type=bind,src=<REDACTED_HOST_PATH>/csci6032-hw2-j0371,dst=/workspace" csci6032-hw2-agent bash -lc "git config --global --add safe.directory /workspace; id; pwd; python --version; git --version; gh --version; copilot version; git status"

# Observed output
uid=1000(agent) gid=1000(agent) groups=1000(agent)
/workspace
Python 3.12.14
git version 2.39.5
gh version 2.101.0 (2026-09-15)
https://github.com/cli/cli/releases/tag/v2.101.0
copilot version 1.2.0 (2025-10-30)
On branch agent-work
Your branch is up to date with 'origin/agent-work'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   AGENTS.md
	modified:   CSCI 6032 Homework 2 AI Usage Notes.ipynb
	modified:   Policies/WORKFLOW_AUDIT.md
	modified:   README.md
	modified:   SavedWorkflowLogs/.ipynb_checkpoints/text-stats-workflow-checkpoint.ipynb
	modified:   SavedWorkflowLogs/text-stats-workflow.ipynb
	modified:   StartNotebook.bat
	modified:   output.txt
	modified:   requirements.txt
	modified:   sample.txt
	modified:   src/text_stats.py
	modified:   tests/test_text_stats.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	Dockerfile

no changes added to commit (use "git add" and/or "git commit -a")


## Step 4: Environment Verification Inside Container

The following commands will be run inside the container to verify the environment:

In [ ]:
# Verification commands and output
# id
# pwd
# python --version
# git --version
# gh --version
# copilot version
# git status

uid=1000(agent) gid=1000(agent) groups=1000(agent)
/workspace
Python 3.12.14
git version 2.39.5
gh version 2.101.0 (2026-09-15)
https://github.com/cli/cli/releases/tag/v2.101.0
copilot version 1.2.0 (2025-10-30)
On branch agent-work
Your branch is up to date with 'origin/agent-work'.


## Step 5: Authentication (Interactive, User-Driven)

### Stop-Point A: GitHub CLI (gh) Authentication

**Instructions for User:**
1. Inside the running container, run: `gh auth login` (or `gh auth login --web`)
2. Complete the browser/device flow authorization
3. Verify authentication with: `gh auth status`
4. Provide redacted confirmation back to the workflow

**Output (to be filled in after user authentication):**

In [ ]:
# gh auth status output (redacted)

### Stop-Point B: GitHub Copilot CLI Authentication

**Instructions for User:**
1. Inside the running container, run: `copilot login`
2. Complete the interactive browser/device flow authorization
3. Verify installation with: `copilot version`
4. Provide confirmation back to the workflow

**Output (to be filled in after user authentication):**

In [ ]:
# copilot version output

### Read-Only GitHub Verification

After both tools are authenticated, run a minimal, read-only verification:

In [ ]:
# gh repo view (redacted output showing repo access confirmed)

## Step 6: Assignment Questions

### Q1: What host resources can the container modify through the bind mount?

**Answer:**

The container can modify any file or directory within the homework repository that is bound at `/workspace`. This includes:
- All source code files (`.py`, `.md`, `.ipynb`, `.txt`, etc.)
- Git tracking data (`.git` directory)
- Subdirectories created by the user (e.g., `src/`, `tests/`, `Workflows/`)
- Any output files or logs generated during development inside the container

The non-root `agent` user runs inside the container with UID 1000 and GID 1000. Because the repo directory is typically owned by a host user with UID 1000 (or the owner has allowed group/other write access), the container's non-root user can read and modify the mounted repository.

### Q2: What important resources did you deliberately not mount?

**Answer:**

The following resources were **intentionally excluded** from the bind mount:

1. **Host Home Directory (`~` or `C:\Users\<USERNAME>`):**
   - Not mounted; container cannot access the user's Documents, Downloads, or other personal directories
   - Reason: Prevents accidental modification or exposure of unrelated user files

2. **Host Credential Directories:**
   - `~/.ssh`: SSH keys and known hosts not exposed
   - `~/.gnupg`: GPG keys not exposed
   - `~/.config`: User configuration and secrets not exposed
   - Reason: Prevents credential leakage and compromised container from accessing authentication material

3. **Browser Profile and Cached Auth:**
   - `~/.mozilla`, `~/.config/google-chrome`, or similar not mounted
   - Reason: Prevents exposure of browser cookies, cached authentication tokens, and browsing history

4. **Docker Socket (`/var/run/docker.sock`):**
   - Docker socket is not exposed to the container
   - Reason: Prevents container from launching sibling containers or escalating privileges

5. **System Configuration (`/etc`, `/sys`, `/proc` beyond read-only kernel views):**
   - Host system directories are not exposed
   - Reason: Prevents container from modifying host system settings or leaking system secrets

6. **Other Parent or Sibling Directories:**
   - Parent directory of the homework repo is not mounted
   - Sibling project directories are not mounted
   - Reason: Limits container scope to only the intended repository, reducing attack surface

**Summary:** Only the homework repository (`/workspace`) is mounted. The container is intentionally isolated from the host's user credentials, system configuration, browser state, Docker daemon, and other project directories.

## Final Verification

**Checklist of Assignment Requirements:**

- [ ] Small, current Linux base (Debian bookworm-slim)
- [ ] Python 3.12 or later installed
- [ ] pip installed
- [ ] Git installed
- [ ] GitHub CLI (gh) installed
- [ ] Current stable GitHub Copilot CLI installed
- [ ] Certificates installed for TLS trust
- [ ] Minimal additional utilities (only curl, wget, ca-certificates)
- [ ] Non-root development user created (agent)
- [ ] /workspace as working directory
- [ ] No repository contents in image
- [ ] No credentials or tokens in image or Dockerfile
- [ ] No browser profile mounted
- [ ] No Docker socket exposed
- [ ] Image tag is exactly `csci6032-hw2-agent`
- [ ] Container runs with repo-only bind mount at /workspace (read/write)
- [ ] All verification commands run and output recorded
- [ ] Container default command is a shell rather than Python REPL, so interactive use behaves as expected
- [ ] gh authenticated interactively (no token pasting)
- [ ] Copilot authenticated interactively (no token pasting)
- [ ] Read-only verification (gh repo view) confirms access